## Notebook18

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
rva = pl.read_csv(ub + "data/flightsrva_flights.csv.gz", null_values=["NA"])
airline = pl.read_csv(ub + "data/flightsrva_airlines.csv.gz", null_values=["NA"])
airport = pl.read_csv(ub + "data/flightsrva_airports.csv.gz", null_values=["NA"])
plane = pl.read_csv(ub + "data/flightsrva_planes.csv.gz", null_values=["NA"])
weather = pl.read_csv(ub + "data/flightsrva_weather.csv.gz", null_values=["NA"])

### Questions

Five tables this time, which is new. Every notebook so far has handed you one table and
asked you to work inside it. This one hands you a small database and asks you to connect it.

`rva` is the table everything else hangs off. It has 24,808 rows, one per commercial flight
that departed Richmond International Airport in 2019, with the scheduled and actual times,
the delays in minutes, the `carrier` that operated it, the `tailnum` of the aircraft, the
`dest` it flew to, and a `time_hour` giving the hour it was scheduled to leave.

The other four are lookup tables. `airline` maps each of the 14 two-letter carrier codes to
a company name. `airport` maps three-letter FAA codes to a name, a latitude and a longitude.
`plane` is a registry of aircraft, one row per tail number, with the manufacturer, the model,
the number of seats and the year it was built. `weather` records conditions at the airport
once an hour, all year.

None of the interesting questions live in one table. The flights table records that 4,258
flights left for `ATL` and has nothing to say about where Atlanta is, who flew there, or what
the weather was doing when they took off.

Unless a question says otherwise, print the result of each block; do not save it.

1. Before joining anything, check that the three lookup tables are what we think they are. A
table is safe to use on the right side of a join when its key has one row per value. Run the
chapter's test on `faa` in `airport`, `carrier` in `airline`, and `tailnum` in `plane`. Then
run the same test on `carrier` in `rva`.

In [ ]:
(
    airport
    .select(c.faa)
    .n_unique() == len(airport)
)

In [ ]:
(
    airline
    .select(c.carrier)
    .n_unique() == len(airline)
)

In [ ]:
(
    plane
    .select(c.tailnum)
    .n_unique() == len(plane)
)

In [ ]:
(
    rva
    .select(c.carrier)
    .n_unique() == len(rva)
)

**Answer**:


That difference is the whole idea. The same column is a primary key in one table and a
foreign key in another, and which one it is depends on the table you are looking at, not on
the values.

2. Join the airline names onto the flights, then report the mean departure delay for each
airline, worst first. Print all 14 rows.

In [ ]:
(
    rva
    .join(airline, on=c.carrier)
    .group_by(c.name)
    .agg(
        n = pl.len(),
        mean_dep_delay = c.dep_delay.mean().round(1)
    )
    .sort(c.mean_dep_delay, descending=True)
    .print(15)
)

**Answer**:


Note the order of operations. The join runs on all 24,808 rows and the grouping happens
afterward. You could group first and join the 14-row summary to `airline` instead, and get
the same answer with less work.

3. The flights table records a destination code and nothing else about it. Join `airport` to
get the real names. The key has a different name in each table: it is `dest` in `rva` and
`faa` in `airport`. Count the flights to each destination, add the mean arrival delay, and
sort by the number of flights.

In [ ]:
(
    rva
    .group_by(c.dest)
    .agg(
        n = pl.len(),
        mean_arr_delay = c.arr_delay.mean().round(1)
    )
    .join(airport, left_on=c.dest, right_on=c.faa)
    .sort(c.n, descending=True)
    .select(c.dest, c.name, c.n, c.mean_arr_delay)
    .print(25)
)

**Answer**:


Here the summary is computed first and the join runs on the 22-row result. That is the right
way around when the lookup adds nothing the grouping needs.

4. The flights table has no coordinates in it. `airport` does. Draw the 22 destinations on a
map: longitude on the x-axis, latitude on the y-axis, sized by the number of flights and
colored by the mean arrival delay.

In [ ]:
(
    rva
    .group_by(c.dest)
    .agg(
        n = pl.len(),
        mean_arr_delay = c.arr_delay.mean()
    )
    .join(airport, left_on=c.dest, right_on=c.faa)
    .pipe(ggplot, aes(c.lon, c.lat))
    .geom_point(aes(color=c.mean_arr_delay, size=c.n))
    .labs(
        x = "Longitude",
        y = "Latitude",
        title = "Where Richmond flew in 2019"
    )
)

**Answer**:


Look at the two tiny dots almost on top of each other at the left edge of the Florida group.
That is Tampa, the darkest point on the whole map at 7.5 minutes early, sitting fifteen miles
from St Petersburg, which is one of the brightest at 15.3 minutes late. Same metro area, two
airports, and no airline flies to both.

This is the ordinary use of a join rather than the exciting one, and it is most of what joins
are for: the coordinates were sitting in a table you were not using.

5. Now use the aircraft registry. Join `plane` to the flights, then report the mean number of
seats per carrier, largest first. Print the whole result.

In [ ]:
(
    rva
    .join(plane, on=c.tailnum)
    .group_by(c.carrier)
    .agg(
        n = pl.len(),
        mean_seats = c.seats.mean().round(1)
    )
    .sort(c.mean_seats, descending=True)
    .print(15)
)

**Answer**:


Count the rows. There are 13, and question 2 said there were 14 airlines. An entire airline
has gone missing from the table, and Polars did not raise an error or print a warning about
it. The join returned 24,346 of the 24,808 flights, and the summary you would have pasted
into a report was computed on the ones that survived.

6. Find out what happened. Use an anti-join to get the flights that have no matching row in
`plane`, and count them by carrier.

In [ ]:
(
    rva
    .join(plane, on=c.tailnum, how="anti")
    .group_by(c.carrier)
    .agg(n = pl.len())
    .sort(c.n, descending=True)
    .print(12)
)

**Answer**:


An anti-join is the right tool here because it hands you the failures themselves. A row count
tells you how many rows you lost. The anti-join tells you which ones, and lets you group them
to find out what they have in common. Here they had one thing in common, and it was an
airline.

7. Two things to check about Allegiant. First, look at the tail numbers `G4` uses. Second,
ask how much of the registry we are actually using: count the planes in `plane` that flew out
of Richmond at least once, with a semi-join.

In [ ]:
(
    rva
    .filter(c.carrier == "G4")
    .select(c.tailnum)
    .unique()
    .head(5)
)

In [ ]:
(
    plane
    .join(rva, on=c.tailnum, how="semi")
    .height
)

**Answer**:


So the obvious fix is to put the `N` back and join again. It matches nothing:

In [ ]:
(
    rva
    .filter(c.carrier == "G4")
    .with_columns(tail_fixed = pl.lit("N") + c.tailnum)
    .join(plane, left_on=c.tail_fixed, right_on=c.tailnum)
    .height
)

The malformed key is a real clue and it is not the cause. Allegiant's aircraft are not in the
registry under any spelling. The semi-join says why that is strange: all 3,120 rows of `plane`
flew out of Richmond, so this is not a national registry that happens to have gaps. It was
built from this very dataset, and Allegiant fell out of whatever process built it, taking its
tail-number formatting with it.

The cost of that inner join is worse than one missing airline. Allegiant is the only carrier
flying to Orlando Sanford, St Petersburg, Sarasota, and Nashville, so an analysis built on
the joined table has four fewer destinations in it. Redraw the map from question 4 that way
and four dots vanish, including the bright one next to Tampa. It would still look like a
perfectly good map.

8. Fix the join. Keep every flight by making it a left join, and give the plane's columns a
suffix so that `year` (the year of the flight) and `year` (the year the aircraft was built)
do not collide. Then report the mean seats and the mean age of the fleet each carrier flew.

In [ ]:
(
    rva
    .join(plane, on=c.tailnum, how="left", suffix="_plane")
    .with_columns(age = 2019 - c.year_plane)
    .group_by(c.carrier)
    .agg(
        n = pl.len(),
        mean_seats = c.seats.mean().round(1),
        mean_age = c.age.mean().round(1)
    )
    .sort(c.mean_age, descending=True)
    .print(15)
)

**Answer**:


The suffix matters more than it looks. Both tables have a column called `year`, and they are
not the same variable: one is 2019 in every row, the other is 1998 or 2007. Without the
suffix Polars still keeps them apart, but it names the second one `year_right`, and
`year_right` is not a name anyone can read three weeks later.

United (`UA`) flies the oldest fleet at 18 years and PSA (`OH`) the newest at 4.2. Spirit
(`NK`) flies the largest aircraft and some of the youngest, which is more or less the
low-cost business model written into a table.

9. On to the weather. Both tables carry a `time_hour` column, which is plainly the intended
way to connect them. The description of this dataset in the book says that `time_hour`
"allows this data to be precisely joined with flight departures." Use it to ask whether
flights leave late when visibility is poor. Then run the same question again, joining on
`year`, `month`, `day`, and `hour` instead, which is a four-column key describing the same
relation. Compare the two.

In [ ]:
(
    rva
    .join(weather.select(c.time_hour, c.visib), on=c.time_hour)
    .with_columns(low_visib = c.visib < 5)
    .group_by(c.low_visib)
    .agg(
        n = pl.len(),
        mean_dep_delay = c.dep_delay.mean().round(1)
    )
    .sort(c.low_visib)
)

In [ ]:
(
    rva
    .join(
        weather.select(c.year, c.month, c.day, c.hour, c.visib),
        on=[c.year, c.month, c.day, c.hour]
    )
    .with_columns(low_visib = c.visib < 5)
    .group_by(c.low_visib)
    .agg(
        n = pl.len(),
        mean_dep_delay = c.dep_delay.mean().round(1)
    )
    .sort(c.low_visib)
)

**Answer**:


10. Work out which key is correct. Print the first scheduled departure of the year and the
weather rows for that morning.

In [ ]:
(
    rva
    .filter((c.month == 1) & (c.day == 1))
    .select(c.sched_dep_time, c.hour, c.time_hour)
    .head(1)
)

In [ ]:
(
    weather
    .filter((c.month == 1) & (c.day == 1) & c.hour.is_in([0, 5]))
    .select(c.hour, c.time_hour, c.visib)
)

**Answer**:


The first flight of 2019 is scheduled for 5:50 in the morning, local time, and its `time_hour`
reads `2019-01-01T05:00:00Z`. In the weather table, local hour 5 has a `time_hour` of
`2019-01-01T10:00:00Z`; the stamp reading `05:00:00Z` belongs to local hour **0**. Richmond is
five hours behind UTC in January, so the weather table is doing the conversion correctly and
the flights table is not: it wrote the local hour down and pasted a `Z` on the end.

So joining on `time_hour` hands every departure the weather from five hours earlier. Our 5:50
am flight was matched to a visibility of 1.0 miles, which was the fog at midnight. Visibility
at 5 am was 10.0 miles. 4,835 of the 24,808 flights get a different visibility depending on
which key you use.

Nothing about this join failed. It matched 24,751 rows and produced a number you would have
believed, in support of a conclusion that happens to be true anyway. The key has the same
name in both tables, the same format and the same type, and the book's own description of the
data recommends it. It is still the wrong key, and the only thing that ever said so was those
two printed rows.

The anti-join leaves the same fingerprint if you look. Under `time_hour`, 57 flights match no
weather at all: 50 on December 31, because the weather feed stops on December 30, and 7 at
6 am on November 3, which is the morning the clocks went back. The weather table's UTC index
has no `06:00:00Z` that day, and the flights table, keeping local time, insists there is one.

One last thing about `weather`, which is the sort of thing people discover after they have
built an analysis on it. The table is mostly empty. `temp`, `dewp`, and `humid` are `null` in
8,678 of the 8,735 rows and `precip` is `null` in 8,467 of them, so wind and visibility are
about all this station recorded. Had we asked our question about temperature instead, the
join would have matched every row it was asked to match and handed back a column of nulls.
That is a quieter failure than the one in question 5 and it comes from the same place: the
join tells you the keys lined up, and nothing else at all.